# S1a. Inspect one sweep run

Minimal notebook for looking at a single window-sweep config yourself. Two knobs:
**`CONFIG`** and **`RESOLUTION`**.

Featurising is the slow step (~1–2 min) and clustering is fast, so they live in
separate cells — change `RESOLUTION` and re-run the clustering cell alone.

For the full four-way comparison see `S1.window_size_sweep.ipynb`.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

# resolve imc_tm from anywhere: repo root, tissuemosaic/, notebooks/, notebooks/archive/
_cands = [c for p in [Path.cwd(), *Path.cwd().parents] for c in (p, p / "tissuemosaic")]
sys.path.insert(0, str(next(c for c in _cands if (c / "imc_tm.py").exists())))
import imc_tm

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.width", 250)

print(imc_tm.sweep_status().to_string(index=False))
print()
print("ckpt_last.pt only appears AFTER a config finishes all 200 epochs.")
print("periodic_ckpts are written every 25 epochs and are usable mid-run.")

## Choose what to look at

`CONFIG` must be one of the names above. Set `USE_PERIODIC` to an epoch number
(e.g. `49`) to inspect a mid-training checkpoint instead of the final one — useful
for a config that is still running.

In [ ]:
CONFIG = "B_fov192_px2.0"  # <-- which run
RESOLUTION = 1.0  # <-- Leiden resolution: higher = more clusters
USE_PERIODIC = None  # <-- None = final ckpt_last.pt, or an epoch int e.g. 49
N_NEIGHBORS = 15
SEED = 0  # <-- pins BOTH the window tiling and Leiden

spec = imc_tm.sweep_spec(CONFIG)
cfg = imc_tm.sweep_config_dict(spec)
print(
    f"{CONFIG}:  FOV {spec['fov_um']} um  |  pixel_size {cfg['pixel_size']} um/px  "
    f"|  global {cfg['global_size']} / local {cfg['local_size']}"
)

if USE_PERIODIC is None:
    ckpt = imc_tm.sweep_checkpoint(CONFIG)
else:
    hits = sorted(
        imc_tm.sweep_run_dir(CONFIG).glob(
            f".neptune/**/periodic_checkpoint-epoch={USE_PERIODIC}.ckpt"
        )
    )
    if not hits:
        avail = sorted(
            int(p.stem.split("=")[-1])
            for p in imc_tm.sweep_run_dir(CONFIG).glob(
                ".neptune/**/periodic_checkpoint-epoch=*.ckpt"
            )
        )
        raise FileNotFoundError(
            f"no periodic checkpoint at epoch {USE_PERIODIC}; available: {avail}"
        )
    ckpt = hits[0]

assert ckpt.is_file(), f"{ckpt} does not exist yet -- is this config trained?"
print(
    "checkpoint:",
    ckpt.relative_to(imc_tm.REPO),
    f"({ckpt.stat().st_size / 1e6:.0f} MB)",
)

## Loss curve

In [ ]:
curve = imc_tm.read_loss_curve(CONFIG)
if len(curve):
    fig, ax = plt.subplots(figsize=(8, 3.6))
    ax.plot(curve.epoch, curve.loss, lw=1.4)
    ax.axvspan(0, cfg["warm_up_epochs"], color="C1", alpha=0.12, label="LR warm-up")
    ax.axvspan(
        cfg["max_epochs"] - cfg["warm_down_epochs"],
        cfg["max_epochs"],
        color="C2",
        alpha=0.12,
        label="LR cosine decay",
    )
    ax.set_xlabel("epoch")
    ax.set_ylabel("DINO loss")
    ax.set_title(CONFIG)
    ax.legend(fontsize=8)
    plt.tight_layout()
    print(
        f"epochs logged: {len(curve)}  |  {curve.loss.iloc[0]:.3f} -> {curve.loss.iloc[-1]:.3f}"
    )
    print(
        "still descending at the end"
        if curve.loss.iloc[-1] < curve.loss.iloc[-20:].mean()
        else "flattening near the end"
    )
else:
    print("no train.log yet")

## Featurise the windows  *(slow cell — ~1–2 min)*

Tiles each ROI into non-overlapping windows and embeds them. Re-run only when you
change `CONFIG`.

`seed` is not cosmetic. TissueMosaic's tiling cropper draws a **random grid origin**
(`i0 = torch.randint(0, stride)` in `dataset.py`) — fine as training augmentation, but
it makes analysis irreproducible: unseeded, three identical calls returned 356 / 345 /
341 windows. `featurize_windows` pins it per ROI, so the same `SEED` gives byte-identical
windows. Change `SEED` to check your conclusions aren't a tiling artefact.

In [ ]:
adatas = {
    r: a
    for r, a in imc_tm.load_all_anndata().items()
    if r.startswith(imc_tm.SWEEP_PATIENT)
}
model = imc_tm.load_model(ckpt)
dm = imc_tm.make_datamodule(cfg)
res = imc_tm.featurize_windows(model, dm, adatas, frac_overlap=0.0, seed=SEED)
del model
print(f"\n{len(res['roi'])} windows total, {res['features'].shape[1]}-dim features")

## Cluster  *(fast cell — re-run this after changing `RESOLUTION`)*

In [ ]:
cl = imc_tm.cluster_embedding(
    res["features"], n_neighbors=N_NEIGHBORS, resolution=RESOLUTION, seed=SEED
)
lab = cl["labels"]
met = imc_tm.window_metrics(res, lab, k=4, n_perm=200, seed=SEED)
print(
    f"resolution {RESOLUTION} -> {met['n_clusters']} clusters over {met['n_windows']} windows\n"
)
for k, v in met.items():
    print(f"  {k:<24s} {v:.4f}" if isinstance(v, float) else f"  {k:<24s} {v}")
print()
print(
    "spatial_coherence_z : >0 means neighbouring windows share a cluster more than chance."
)
print(
    "roi_mixing          : ~null (~0.97) = clusters span ROIs; near 0 = clusters ARE the ROIs."
)
print(
    "composition_silhouette : how much of the embedding is explained by bulk composition"
)
print("                         ALONE. Low/negative means the model keyed on spatial")
print("                         ARRANGEMENT rather than proportions -- not a failure.")

## UMAP

Two views of the same embedding. The **ROI** panel is the failure check: if the ROIs
separate cleanly there, the model keyed on batch rather than on tissue motifs.

In [ ]:
# one colour per cluster, shared by every figure below
cmap_cl = plt.get_cmap("tab20")
CLUSTER_COLORS = {c: cmap_cl(i % 20) for i, c in enumerate(sorted(np.unique(lab)))}
ROIS = sorted(adatas)
cmap_roi = plt.get_cmap("tab10")
ROI_COLORS = {r: cmap_roi(i) for i, r in enumerate(ROIS)}

fig, axes = plt.subplots(1, 2, figsize=(15, 6.2))

for c in sorted(np.unique(lab)):
    m = lab == c
    axes[0].scatter(
        cl["umap"][m, 0],
        cl["umap"][m, 1],
        color=CLUSTER_COLORS[c],
        s=26,
        linewidths=0,
        label=f"cluster {c}  (n={m.sum()})",
    )
axes[0].set_title(
    f"Leiden cluster  —  resolution {RESOLUTION}, {met['n_clusters']} clusters"
)
axes[0].legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8, frameon=False)

for r in ROIS:
    m = res["roi"] == r
    axes[1].scatter(
        cl["umap"][m, 0],
        cl["umap"][m, 1],
        color=ROI_COLORS[r],
        s=26,
        linewidths=0,
        label=f"{r}  (n={m.sum()})",
    )
axes[1].set_title(f"ROI  —  roi_mixing {met['roi_mixing']:.3f} (null ~0.97)")
axes[1].legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8, frameon=False)

for a in axes:
    a.set_xticks([])
    a.set_yticks([])
    a.set_xlabel("UMAP-1")
    a.set_ylabel("UMAP-2")
plt.suptitle(
    f"{CONFIG} — {len(lab)} windows, FOV {spec['fov_um']} um", y=1.01, fontsize=13
)
plt.tight_layout()

## Clusters in tissue

Each square is one window, at its true position. **y is up** — the ICS frame is already
in anatomical orientation. Contiguous blocks of one colour mean the window size resolves
real tissue domains; speckle means it does not.

In [ ]:
ncol = len(ROIS)
fig, axes = plt.subplots(1, ncol, figsize=(5.0 * ncol, 5.4))
axes = np.atleast_1d(axes)
for ax, r in zip(axes, ROIS):
    m = res["roi"] == r
    ax.scatter(
        res["centers_um"][m, 0],
        res["centers_um"][m, 1],
        color=[CLUSTER_COLORS[c] for c in lab[m]],
        s=250,
        marker="s",
        linewidths=0,
    )
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f"{r}  ({m.sum()} windows)")

handles = [
    plt.Line2D(
        [0],
        [0],
        marker="s",
        linestyle="",
        markersize=9,
        color=CLUSTER_COLORS[c],
        label=f"cluster {c}",
    )
    for c in sorted(np.unique(lab))
]
fig.legend(
    handles=handles,
    bbox_to_anchor=(1.005, 0.5),
    loc="center left",
    fontsize=9,
    frameon=False,
)
plt.suptitle(
    f"{CONFIG} — cluster per window, tissue coordinates (y up).  "
    f"spatial_coherence z = {met['spatial_coherence_z']:.1f}",
    y=1.02,
    fontsize=13,
)
plt.tight_layout()

## What is each cluster made of?

Mean cell-type fraction per cluster. The `ALL` row is the cohort mean — a cluster is
interesting where it departs from it.

In [ ]:
comp = pd.DataFrame(res["composition"], columns=imc_tm.CELL_TYPES)
comp["cluster"] = lab
prof = comp.groupby("cluster")[imc_tm.CELL_TYPES].mean()
prof.insert(0, "n_windows", comp.groupby("cluster").size())
prof.loc["ALL"] = [len(lab)] + list(res["composition"].mean(axis=0))
display(
    prof.round(3).style.background_gradient(
        axis=0, cmap="Blues", subset=imc_tm.CELL_TYPES
    )
)

### The same thing as stacked bars

Left: all ten channels. `other_tissue` is ~67% of every window, so it swamps the stack —
which is exactly why the right panel drops the two background channels and renormalises
to the classified biology. `ALL` is the cohort mean, for reference.

In [ ]:
cell_colors = imc_tm.palette()
prof_bar = prof.drop(index="ALL")[imc_tm.CELL_TYPES]
prof_bar.loc["ALL"] = res["composition"].mean(axis=0)

bio = prof_bar[imc_tm.BIOLOGY_TYPES]
bio = bio.div(bio.sum(axis=1), axis=0)

fig, axes = plt.subplots(1, 2, figsize=(17, 5.4))
for ax, (tbl, cols, title) in zip(
    axes,
    [
        (prof_bar, imc_tm.CELL_TYPES, "all 10 channels"),
        (bio, imc_tm.BIOLOGY_TYPES, "classified biology only, renormalised"),
    ],
):
    bottom = np.zeros(len(tbl))
    x = np.arange(len(tbl))
    for t in cols:
        ax.bar(
            x, tbl[t].values, bottom=bottom, color=cell_colors[t], label=t, width=0.82
        )
        bottom += tbl[t].values
    ax.set_xticks(x)
    ax.set_xticklabels(
        [f"c{i}" if i != "ALL" else "ALL" for i in tbl.index], fontsize=9
    )
    ax.set_xlabel("cluster")
    ax.set_ylabel("mean fraction of cells")
    ax.set_ylim(0, 1)
    ax.set_title(title)
    # mark the cohort-mean reference bar
    ax.axvline(len(tbl) - 1.5, color="0.3", lw=1, ls=":")
# ONE legend covering all ten channels -- the left panel's two dominant colours
# (other_tissue, duct_filler) are absent from the right panel's stack, so a
# biology-only legend would leave them unlabelled.
handles = [
    plt.Rectangle(
        (0, 0),
        1,
        1,
        color=cell_colors[t],
        label=t + ("  (background)" if t in imc_tm.BACKGROUND_TYPES else ""),
    )
    for t in imc_tm.CELL_TYPES
]
fig.legend(
    handles=handles,
    bbox_to_anchor=(1.005, 0.5),
    loc="center left",
    fontsize=9,
    frameon=False,
    title="cell type",
)
plt.suptitle(f"{CONFIG} — cell type composition per cluster", y=1.02, fontsize=13)
plt.tight_layout()

### Which ROI does each cluster come from?

In [ ]:
ct = pd.crosstab(lab, res["roi"])
ct["total"] = ct.sum(axis=1)
print(ct.to_string())
print()
top = comp.groupby("cluster")[imc_tm.BIOLOGY_TYPES].mean().idxmax(axis=1)
print("dominant BIOLOGY channel per cluster (background excluded):")
for c, t in top.items():
    frac = comp.groupby("cluster")[t].mean()[c]
    enrich = frac / max(res["composition"][:, imc_tm.CELL_TYPES.index(t)].mean(), 1e-9)
    print(f"  cluster {c:<3} {t:<34s} {frac:.3f}   ({enrich:.1f}x cohort mean)")

## Notes

- **`USE_PERIODIC`** lets you inspect a config that is still training — set it to any
  epoch that has a periodic checkpoint (every 25). Useful for watching a run develop.
- **Window counts are set by FOV, not resolution.** The 192 µm configs (B, C, D) all
  tile into a similar number of windows despite very different rasters; A at 128 µm
  yields roughly twice as many.
- This notebook deliberately does **no** cross-config comparison — for that use
  `S1.window_size_sweep.ipynb`, which runs all four with identical clustering settings
  and permutation-based metrics.